# Notebook 07: PagedAttention

## Why This Matters
The KV cache is the memory bottleneck in LLM serving. Traditional systems pre-allocate contiguous memory for each request's maximum possible sequence length — but most requests don't reach the maximum, so memory is wasted. Worse, when multiple requests finish at different times, the freed memory is fragmented and can't be reused efficiently.

PagedAttention (Kwon et al., 2023) — the core innovation in vLLM — borrows virtual memory paging from OS design. It divides the KV cache into fixed-size pages (blocks) that can be allocated non-contiguously, eliminating fragmentation and enabling 2–4× higher serving throughput.

## Structure
1. The memory fragmentation problem in LLM serving
2. Virtual memory paging — the OS analogy
3. PagedAttention: KV blocks and block tables
4. Implementation: the block manager
5. Copy-on-write for parallel sampling
6. Memory utilization comparison
7. Throughput impact

## What You'll Learn
- Why contiguous KV cache allocation is inherently wasteful
- How block tables map logical to physical KV positions
- How copy-on-write enables beam search without memory duplication
- The memory utilization numbers that motivated vLLM

## Background

### The serving memory problem

Once you have a working LLM and want to serve it to real users, you encounter a new class of problem: how do you efficiently manage memory for many concurrent requests? Each request needs its own KV cache, and requests finish at different times with unpredictable output lengths.

The naive approach — pre-allocate the maximum possible KV cache for each request upfront — wastes enormous amounts of memory. A request that generates 100 tokens but was allocated memory for 2048 wastes 95% of its allocation. When 20 requests are running simultaneously, this waste compounds: you're holding 20× max-length allocations even though average actual usage is a fraction of that.

Early serving systems (Orca, 2022; early HuggingFace TGI) achieved only 40–60% KV cache utilization because of this fragmentation. The remaining 40–60% of GPU memory sat allocated but unused while users waited for capacity.

### The OS analogy that inspired vLLM

Woosuk Kwon and colleagues at UC Berkeley (2023) recognized that LLM serving had the same memory management problem that operating systems solved decades ago with **virtual memory paging**.

In OS design, processes don't access physical RAM directly. Each process sees a virtual address space that appears contiguous, but the OS maps virtual pages to non-contiguous physical page frames using a page table. Memory is allocated on demand, one page at a time. When a process ends, its pages are returned to the pool. Copy-on-write lets two processes share physical pages until one of them writes — at which point only the modified page is copied.

PagedAttention applies this exactly: divide the KV cache into fixed-size **blocks** (e.g., 16 tokens each). Each request has a **block table** that maps its logical block indices to physical block locations in GPU memory. Blocks are allocated on demand as tokens are generated, freed immediately when a request completes, and can be placed non-contiguously — no fragmentation.

### vLLM and the throughput results

vLLM (2023) was the first production serving system to implement PagedAttention. The throughput results were striking: 2–4× higher request throughput compared to Orca and TGI at the time, primarily by increasing KV cache utilization from ~50% to >95%.

The intuition for why utilization → throughput: more efficient memory use means more concurrent requests fit in GPU memory simultaneously. More concurrent requests means larger effective batch sizes during the decode phase. Larger decode batches amortize the memory-bandwidth cost of reading the KV cache across more tokens per second. The whole chain follows from fixing memory fragmentation.

vLLM has since become one of the most widely used open-source LLM serving systems, and PagedAttention is the memory management approach used (with variations) by TGI, TensorRT-LLM, and most production-grade serving stacks.

In [ ]:
%pip install -q torch

In [ ]:
import math
import random
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import torch

random.seed(42)
print("Imports OK")

## 1. The Memory Fragmentation Problem

Traditional LLM serving pre-allocates KV cache per request:

```
GPU memory (24 GB total, 14 GB weights, 10 GB for KV cache)

Request A: max_seq=2048 → allocate 1 GB contiguous
Request B: max_seq=2048 → allocate 1 GB contiguous
Request C: max_seq=2048 → allocate 1 GB contiguous
...

Reality:
  Request A generates 100 tokens, then finishes → 900 MB WASTED
  Request B generates 50 tokens, then finishes  → 950 MB WASTED
  Request C generates 2048 tokens (rare case)   → 0 MB wasted

Problems:
1. Internal fragmentation: allocated but unused space within each request
2. External fragmentation: finished requests leave non-contiguous holes
3. Reserved memory: systems reserve max_seq upfront — can't serve more requests
```

The Orca paper (2022) showed that real serving workloads waste 60–80% of KV cache memory this way.

In [ ]:
# Simulate traditional contiguous allocation
def simulate_contiguous_allocation(
    requests: List[int],   # actual output lengths
    max_seq: int = 2048,
    bytes_per_token: int = 1_048_576,  # 1 MB per token (LLaMA-7B at 1 seq)
) -> dict:
    """Simulate traditional KV cache allocation and measure waste."""
    total_allocated = len(requests) * max_seq * bytes_per_token
    total_used = sum(r * bytes_per_token for r in requests)
    wasted = total_allocated - total_used

    return {
        "n_requests": len(requests),
        "total_allocated_gb": total_allocated / 1e9,
        "total_used_gb": total_used / 1e9,
        "wasted_gb": wasted / 1e9,
        "utilization": total_used / total_allocated,
        "avg_output_len": sum(requests) / len(requests),
    }


# Real-world output length distributions
import random
random.seed(42)

# Short-tail: most requests are short (chat API)
chat_requests = [random.randint(20, 300) for _ in range(20)]

# Mixed: some short, some long (general API)
mixed_requests = [random.choice(
    [random.randint(10, 100),    # 60% short
     random.randint(100, 500),   # 30% medium
     random.randint(500, 2048)]  # 10% long
) for _ in range(20)]

for name, reqs in [("Chat (short outputs)", chat_requests),
                    ("Mixed workload", mixed_requests)]:
    stats = simulate_contiguous_allocation(reqs)
    print(f"{name}:")
    print(f"  Avg output length:  {stats['avg_output_len']:.0f} tokens")
    print(f"  Allocated:          {stats['total_allocated_gb']:.2f} GB")
    print(f"  Actually used:      {stats['total_used_gb']:.2f} GB")
    print(f"  Wasted:             {stats['wasted_gb']:.2f} GB ({(1-stats['utilization'])*100:.1f}%)")
    print()

## 2. The OS Virtual Memory Analogy

Operating systems solved an identical problem decades ago with **virtual memory paging**:

```
OS Virtual Memory:              PagedAttention:
─────────────────               ──────────────
Process address space     →     Request's KV sequence
Virtual page              →     Logical KV block
Physical page frame       →     Physical KV block (GPU memory)
Page table                →     Block table
Page fault                →     Block allocation on demand
Copy-on-write (fork)      →     Copy-on-write (parallel sampling)
```

Key insight: processes (requests) see a contiguous virtual address space, but physical memory is fragmented into non-contiguous pages (blocks). A lookup table maps virtual → physical.

PagedAttention applies this to KV cache:
- Divide KV cache into fixed-size **blocks** (e.g., 16 tokens each)
- Each request has a **block table** mapping logical block index → physical block
- Blocks are allocated on demand, freed when request completes
- Non-contiguous physical blocks → no fragmentation

## 3. PagedAttention: KV Blocks and Block Tables

In [ ]:
@dataclass
class KVBlock:
    """A physical block of KV cache memory."""
    block_id: int
    block_size: int           # tokens per block
    n_layers: int
    d_model: int
    ref_count: int = 0        # how many sequences reference this block
    is_full: bool = False
    tokens_used: int = 0

    def memory_bytes(self, dtype_bytes: int = 2) -> int:
        """Memory footprint of this block."""
        return 2 * self.n_layers * self.block_size * self.d_model * dtype_bytes


@dataclass
class Request:
    """An active inference request."""
    request_id: str
    block_table: List[int] = field(default_factory=list)  # logical → physical block IDs
    tokens_generated: int = 0

    def logical_block_for_token(self, token_pos: int, block_size: int) -> int:
        return token_pos // block_size


class BlockManager:
    """
    PagedAttention block manager.
    Manages a pool of physical KV cache blocks.
    """
    def __init__(
        self,
        total_memory_gb: float,
        block_size: int,
        n_layers: int,
        d_model: int,
        dtype_bytes: int = 2,
    ):
        self.block_size = block_size
        self.n_layers = n_layers
        self.d_model = d_model

        # Calculate how many blocks fit in memory
        bytes_per_block = 2 * n_layers * block_size * d_model * dtype_bytes
        self.num_blocks = int(total_memory_gb * 1e9 / bytes_per_block)

        # Free block pool
        self.free_blocks: List[int] = list(range(self.num_blocks))
        self.physical_blocks: Dict[int, KVBlock] = {
            i: KVBlock(i, block_size, n_layers, d_model)
            for i in range(self.num_blocks)
        }

        self.requests: Dict[str, Request] = {}
        self._alloc_events: List[str] = []

    def allocate_block(self, request_id: str) -> Optional[int]:
        """Allocate a physical block for a request. Returns block_id or None."""
        if not self.free_blocks:
            return None  # OOM — need to evict
        block_id = self.free_blocks.pop(0)
        self.physical_blocks[block_id].ref_count = 1
        self.physical_blocks[block_id].tokens_used = 0
        self._alloc_events.append(f"Allocated block {block_id} → {request_id}")
        return block_id

    def free_blocks_for_request(self, request_id: str) -> None:
        """Free all blocks owned by a request."""
        if request_id not in self.requests:
            return
        req = self.requests[request_id]
        for block_id in req.block_table:
            block = self.physical_blocks[block_id]
            block.ref_count -= 1
            if block.ref_count == 0:
                self.free_blocks.append(block_id)
                self._alloc_events.append(f"Freed block {block_id} from {request_id}")
        del self.requests[request_id]

    def append_token(self, request_id: str) -> Tuple[int, int]:
        """
        Append a new token to a request's KV cache.
        Allocates a new block if needed.
        Returns (logical_block, physical_block).
        """
        req = self.requests[request_id]
        token_pos = req.tokens_generated
        logical_block = token_pos // self.block_size

        # Need a new block?
        if logical_block >= len(req.block_table):
            new_block = self.allocate_block(request_id)
            if new_block is None:
                raise RuntimeError(f"OOM: no free blocks for {request_id}")
            req.block_table.append(new_block)

        physical_block = req.block_table[logical_block]
        self.physical_blocks[physical_block].tokens_used += 1
        req.tokens_generated += 1

        return logical_block, physical_block

    def add_request(self, request_id: str) -> None:
        self.requests[request_id] = Request(request_id)

    @property
    def free_block_count(self):
        return len(self.free_blocks)

    @property
    def utilization(self):
        used = self.num_blocks - len(self.free_blocks)
        return used / self.num_blocks


# Demonstrate block manager
mgr = BlockManager(
    total_memory_gb=2.0,
    block_size=16,         # 16 tokens per block
    n_layers=32,
    d_model=4096,
    dtype_bytes=2,
)

print(f"Block manager initialized:")
print(f"  Total physical blocks: {mgr.num_blocks}")
print(f"  Block size: {mgr.block_size} tokens")
print(f"  Memory per block: {2 * 32 * 16 * 4096 * 2 / 1e6:.2f} MB")
print(f"  Total capacity: {mgr.num_blocks * mgr.block_size} tokens")

In [ ]:
# Simulate serving multiple requests
print("Simulating serving 4 requests with different lengths:")
print()

# Add requests
request_lengths = {"req-A": 45, "req-B": 20, "req-C": 70, "req-D": 10}
for req_id in request_lengths:
    mgr.add_request(req_id)

# Generate tokens for all requests simultaneously (simulating continuous batching)
max_len = max(request_lengths.values())
for step in range(max_len):
    for req_id, length in request_lengths.items():
        if req_id in mgr.requests and mgr.requests[req_id].tokens_generated < length:
            logical, physical = mgr.append_token(req_id)

    # Check for finished requests
    for req_id, length in list(request_lengths.items()):
        if req_id in mgr.requests and mgr.requests[req_id].tokens_generated >= length:
            req = mgr.requests[req_id]
            print(f"  Step {step+1:3d}: {req_id} finished ({length} tokens, "
                  f"{len(req.block_table)} blocks used)")
            print(f"           Block table: {req.block_table}")
            mgr.free_blocks_for_request(req_id)
            print(f"           Freed → {mgr.free_block_count} blocks now available")
            print()

print(f"Final state: {mgr.free_block_count}/{mgr.num_blocks} blocks free")

## 4. Block Table: Logical → Physical Mapping

During attention computation, the block table translates logical token positions to physical GPU memory locations:

```
Request A, token_pos=37, block_size=16:

  logical_block = 37 // 16 = 2
  offset_within_block = 37 % 16 = 5

  block_table[2] = 47  (physical block 47)

  KV cache address = kv_cache[47, :, 5, :]  ← non-contiguous with blocks 0 and 1!
```

The attention kernel must follow these indirections. The PagedAttention CUDA kernel was written specifically to handle non-contiguous KV blocks.

In [ ]:
def paged_attention_lookup(
    token_positions: List[int],
    block_table: List[int],
    block_size: int,
) -> List[Tuple[int, int]]:
    """
    Map token positions to (physical_block, offset) pairs.
    This is what the attention kernel does to gather KV entries.
    """
    results = []
    for pos in token_positions:
        logical_block = pos // block_size
        offset = pos % block_size
        physical_block = block_table[logical_block]
        results.append((physical_block, offset))
    return results


# Example: a 50-token sequence with block_size=16
# The 4 blocks are scattered across non-contiguous physical memory
example_block_table = [7, 23, 51, 4]  # logical blocks 0,1,2,3 → physical 7,23,51,4

print("Block table lookup (logical → physical):")
print(f"Block table: {example_block_table} (block_size=16)")
print()
print(f"{'Token pos':>10} | {'Logical block':>14} | {'Physical block':>15} | {'Offset':>8}")
print("-" * 55)
for pos in [0, 15, 16, 31, 32, 47, 48, 49]:
    lb = pos // 16
    pb = example_block_table[lb]
    off = pos % 16
    print(f"{pos:>10} | {lb:>14} | {pb:>15} | {off:>8}")

print()
print("Note: Positions 0-15 are in physical block 7,")
print("      positions 16-31 in physical block 23 — non-contiguous in GPU memory")

## 5. Copy-on-Write for Parallel Sampling

Beam search and parallel sampling generate multiple sequences from the same prompt. Without copy-on-write, you'd need to duplicate the KV cache for each beam — expensive.

With copy-on-write:
```
Prompt: "The cat" → KV cache blocks [7, 23] (shared)

Beam 1 fork: block_table = [7, 23, ...new...]  ref_count[7]++, ref_count[23]++
Beam 2 fork: block_table = [7, 23, ...new...]  ref_count[7]++, ref_count[23]++

When Beam 1 writes to block 23: ref_count[23] > 1 → copy block 23 to new block 61
                                  Beam 1 block_table: [7, 61, ...]
                                  ref_count[23]--, ref_count[61]=1

Beam 2 still reads from block 23 (unmodified)
```

Prompt tokens are shared across all beams — only divergent tokens need their own blocks.

In [ ]:
class CopyOnWriteBlockManager(BlockManager):
    """Extends BlockManager with copy-on-write for parallel sampling."""

    def fork_request(self, source_id: str, new_id: str) -> None:
        """
        Fork a request (for beam search / parallel sampling).
        The new request shares all blocks with the source — no copy yet.
        """
        source = self.requests[source_id]
        # Share all blocks: increment ref counts
        for block_id in source.block_table:
            self.physical_blocks[block_id].ref_count += 1
        # New request shares the same block table
        self.requests[new_id] = Request(
            request_id=new_id,
            block_table=source.block_table.copy(),
            tokens_generated=source.tokens_generated,
        )
        self._alloc_events.append(f"Forked {source_id} → {new_id} (sharing {len(source.block_table)} blocks)")

    def cow_append_token(self, request_id: str) -> Tuple[int, int]:
        """
        Append token with copy-on-write: if the last block is shared, copy it first.
        """
        req = self.requests[request_id]
        token_pos = req.tokens_generated
        logical_block = token_pos // self.block_size

        if logical_block < len(req.block_table):
            # Writing to an existing block — check if shared
            physical_block = req.block_table[logical_block]
            if self.physical_blocks[physical_block].ref_count > 1:
                # Copy-on-write: allocate new block, copy data
                new_block_id = self.allocate_block(request_id)
                self._alloc_events.append(
                    f"COW: copied block {physical_block} → {new_block_id} for {request_id}"
                )
                self.physical_blocks[physical_block].ref_count -= 1
                req.block_table[logical_block] = new_block_id
                physical_block = new_block_id

        return self.append_token(request_id)


# Demonstrate copy-on-write with beam search
mgr_cow = CopyOnWriteBlockManager(
    total_memory_gb=1.0, block_size=16, n_layers=32, d_model=4096
)

# Process a shared prompt
mgr_cow.add_request("prompt")
for _ in range(32):  # 32 tokens = 2 full blocks
    mgr_cow.append_token("prompt")

prompt_blocks = mgr_cow.requests["prompt"].block_table.copy()
print(f"After prompt (32 tokens):")
print(f"  Blocks used: {prompt_blocks}")
print(f"  Ref counts: {[mgr_cow.physical_blocks[b].ref_count for b in prompt_blocks]}")
print()

# Fork into 3 beams
mgr_cow.fork_request("prompt", "beam-1")
mgr_cow.fork_request("prompt", "beam-2")
mgr_cow.fork_request("prompt", "beam-3")

print(f"After forking into 3 beams:")
for b in prompt_blocks:
    print(f"  Block {b}: ref_count={mgr_cow.physical_blocks[b].ref_count}")
print(f"  Free blocks remaining: {mgr_cow.free_block_count}")
print()

# Each beam generates divergent tokens
for beam in ["beam-1", "beam-2", "beam-3"]:
    for _ in range(8):  # 8 tokens, fills half a new block
        mgr_cow.cow_append_token(beam)

print("After 8 divergent tokens per beam:")
for beam in ["beam-1", "beam-2", "beam-3"]:
    bt = mgr_cow.requests[beam].block_table
    print(f"  {beam}: blocks={bt}  (first 2 still shared)")

## 6. Memory Utilization Comparison

In [ ]:
def simulate_paged_utilization(
    requests: List[int],
    block_size: int = 16,
) -> dict:
    """Simulate PagedAttention memory utilization."""
    # Blocks are allocated as needed, freed when done
    # Worst case waste: last block of each request is partially filled
    total_tokens = sum(requests)

    # Each request wastes at most (block_size - 1) tokens in its last block
    total_blocks_used = sum(math.ceil(r / block_size) for r in requests)
    total_slots = total_blocks_used * block_size
    wasted_slots = total_slots - total_tokens

    return {
        "total_tokens": total_tokens,
        "total_blocks": total_blocks_used,
        "total_slots": total_slots,
        "wasted_slots": wasted_slots,
        "utilization": total_tokens / total_slots,
        "max_waste_per_request": block_size - 1,
    }


print("Memory utilization comparison:")
print(f"{'Method':<25} {'Utilization':>12} {'Waste'}")  
print("-" * 55)

for name, reqs in [("Chat requests", chat_requests), ("Mixed workload", mixed_requests)]:
    print(f"\nWorkload: {name} (avg={sum(reqs)/len(reqs):.0f} tokens)")

    # Contiguous
    cont = simulate_contiguous_allocation(reqs, max_seq=2048, bytes_per_token=1)
    print(f"  Contiguous alloc:       {cont['utilization']:>10.1%}  ({(1-cont['utilization'])*100:.1f}% wasted)")

    # Paged (block_size=16)
    paged_16 = simulate_paged_utilization(reqs, block_size=16)
    print(f"  PagedAttention (b=16):  {paged_16['utilization']:>10.1%}  ({(1-paged_16['utilization'])*100:.1f}% wasted)")

    # Paged (block_size=4) — smaller blocks, less internal fragmentation
    paged_4 = simulate_paged_utilization(reqs, block_size=4)
    print(f"  PagedAttention (b=4):   {paged_4['utilization']:>10.1%}  ({(1-paged_4['utilization'])*100:.1f}% wasted)")

## 7. Throughput Impact

The vLLM paper (Kwon et al., 2023) reported throughput improvements vs. prior systems:

```
System          Memory utilization    Throughput (tokens/s)
Orca            ~40–60%               baseline
HuggingFace TGI ~40–60%               similar to Orca  
vLLM (Paged)    ~95%+                 2–4× Orca
```

Higher memory utilization = more requests fit in GPU memory simultaneously = higher batch sizes = better GPU utilization during decode = more throughput.

The key chain of reasoning:
```
PagedAttention → higher KV cache utilization
              → more concurrent requests per GPU
              → larger effective batch size
              → better amortization of memory bandwidth during decode
              → higher throughput
```

In [ ]:
# Model the throughput benefit from higher concurrency

def model_throughput(
    gpu_memory_gb: float,
    model_weights_gb: float,
    kv_cache_utilization: float,
    bytes_per_token_kv: float,
    avg_output_len: int,
    tokens_per_second_per_request: float = 50.0,
) -> dict:
    """Estimate throughput given memory utilization."""
    available_for_kv = gpu_memory_gb - model_weights_gb
    usable_kv = available_for_kv * kv_cache_utilization
    max_concurrent = usable_kv * 1e9 / (avg_output_len * bytes_per_token_kv)

    # Throughput scales with batch size up to a point
    # Simplified: linear scaling with concurrency (reality is sublinear)
    throughput = max_concurrent * tokens_per_second_per_request

    return {
        "max_concurrent_requests": int(max_concurrent),
        "estimated_throughput_tps": int(throughput),
        "kv_cache_gb": usable_kv,
    }


# LLaMA-7B on A100 80GB
gpu_gb = 80.0
weights_gb = 14.0
bytes_per_token = 1_048_576 / 2048  # ~512 KB per token for LLaMA-7B

systems = [
    ("Traditional (40% util)",     0.40),
    ("Improved (60% util)",         0.60),
    ("PagedAttention (95% util)",   0.95),
]

print("Throughput model (LLaMA-7B, A100 80GB, avg output=256 tokens):")
print(f"{'System':<30} {'KV Cache (GB)':>14} {'Max Concurrent':>16} {'Throughput (tps)':>18}")
print("-" * 80)
for name, util in systems:
    stats = model_throughput(gpu_gb, weights_gb, util, bytes_per_token, 256)
    print(f"{name:<30} {stats['kv_cache_gb']:>14.1f} {stats['max_concurrent_requests']:>16} "
          f"{stats['estimated_throughput_tps']:>18,}")

## Summary

```
Problem: Traditional contiguous KV cache allocation wastes 40-80% of GPU memory
Cause:   Pre-allocation to max_seq_len; fragmentation when requests finish

Solution: PagedAttention
  - Divide KV cache into fixed-size blocks (e.g., 16 tokens each)
  - Allocate blocks on demand, not upfront
  - Block table maps logical → physical blocks (like OS page tables)
  - Copy-on-write enables beam search without memory duplication

Result:
  - Memory utilization: 40% → 95%+
  - Max concurrent requests: 2-4× more
  - Throughput: 2-4× higher
```

**Block size tradeoff:** Smaller blocks → less internal fragmentation, but more block table overhead and more non-contiguous memory accesses. vLLM defaults to 16 tokens per block.

**Next:** Notebook 08 covers prefix caching — reusing KV cache across requests that share a common prompt prefix, cutting TTFT for repeated system prompts.